# Aula 8 — IA Generativa e Prompt Engineering

**Notebook-laboratório do estudante**

Nesta prática, você criará e avaliará prompts sem precisar de chave de API. Os prompts produzidos podem ser copiados para a ferramenta autorizada pela instituição.

> Nunca envie dados pessoais ou sensíveis reais nos testes da aula.


## Objetivos

Diferenciar regra, ML e IA generativa; estruturar prompts com papel, tarefa, contexto, regras e formato; reduzir alucinações; documentar V1, V2 e V3; planejar validação humana e privacidade.


In [ ]:
import io
import os
import pandas as pd
from IPython.display import display

print("Laboratório preparado!")


In [ ]:
def carregar_csv(nome_arquivo):
    if os.path.exists(nome_arquivo):
        return pd.read_csv(nome_arquivo)
    try:
        from google.colab import files
        print(f"Selecione: {nome_arquivo}")
        enviados = files.upload()
        nome = nome_arquivo if nome_arquivo in enviados else next(iter(enviados))
        return pd.read_csv(io.BytesIO(enviados[nome]))
    except ImportError as erro:
        raise FileNotFoundError(f"Coloque '{nome_arquivo}' junto ao notebook.") from erro


## 1. IA tradicional × IA generativa

| Abordagem | Entrada | Saída típica |
|---|---|---|
| Regra | condição explícita | ação determinística |
| Machine Learning | features | classe ou valor previsto |
| IA generativa | instrução + contexto | conteúdo gerado |

Uma agenda pode usar regra para conflitos, ML para prever prioridade e IA generativa para redigir um resumo.


## 2. Conhecendo os casos de teste


In [ ]:
casos = carregar_csv("casos_prompt_aula08.csv")
print("Casos disponíveis:", len(casos))
display(casos)


## 3. Prompt fraco

`Faça um resumo.`

Ele não define dados, limites, público nem formato. Vamos comparar com uma versão estruturada.


In [ ]:
prompt_fraco = "Faça um resumo."
print(prompt_fraco)


## 4. Gerador de prompt estruturado


In [ ]:
def montar_prompt(papel, tarefa, dados, regras, formato):
    return f"""PAPEL:
{papel}

TAREFA:
{tarefa}

CONTEXTO / DADOS:
{dados}

REGRAS:
{regras}
Use somente os dados fornecidos.
Se a informação solicitada não estiver nos dados, responda que não há informação suficiente.

FORMATO:
{formato}"""


caso = casos.iloc[0]
prompt_estruturado = montar_prompt(
    papel="Você é um assistente de organização que prepara rascunhos para revisão humana.",
    tarefa=caso["tarefa"],
    dados=caso["dados"],
    regras=caso["regra_seguranca"],
    formato=caso["formato"],
)
print(prompt_estruturado)


## 5. Checklist automático introdutório

Este verificador não garante qualidade; ele apenas procura componentes explícitos. A avaliação humana continua indispensável.


In [ ]:
def verificar_componentes(prompt):
    componentes = ["PAPEL:", "TAREFA:", "CONTEXTO / DADOS:", "REGRAS:", "FORMATO:"]
    resultado = [{"componente": item.replace(":", ""), "presente": item in prompt} for item in componentes]
    resultado.append({"componente": "Regra contra invenção", "presente": "somente os dados" in prompt.lower()})
    resultado.append({"componente": "Resposta para ausência", "presente": "informação suficiente" in prompt.lower()})
    return pd.DataFrame(resultado)

display(verificar_componentes(prompt_fraco))
display(verificar_componentes(prompt_estruturado))


## 6. Escolhendo um caso para o grupo


In [ ]:
# Altere para um valor entre C01 e C12.
ID_CASO = "C03"
caso_grupo = casos.loc[casos["id_caso"] == ID_CASO].iloc[0]
display(caso_grupo.to_frame("valor"))


## 7. Três versões do prompt

A V1 será propositalmente simples. A V2 adiciona estrutura. A V3 acrescenta controles contra invenção e revisão humana.


In [ ]:
prompt_v1 = f"{caso_grupo['tarefa']}. Dados: {caso_grupo['dados']}"

prompt_v2 = montar_prompt(
    papel="Você é um assistente que organiza informações do projeto.",
    tarefa=caso_grupo["tarefa"],
    dados=caso_grupo["dados"],
    regras=caso_grupo["regra_seguranca"],
    formato=caso_grupo["formato"],
)

prompt_v3 = prompt_v2 + """

VALIDAÇÃO:
Apresente a saída como RASCUNHO.
Inclua uma seção chamada INFORMAÇÕES AUSENTES.
Não transforme ausência de dados em suposição.
A resposta deverá ser revisada por uma pessoa antes de ser utilizada."""

print("PROMPT V1\n", prompt_v1)
print("\nPROMPT V2\n", prompt_v2)
print("\nPROMPT V3\n", prompt_v3)


In [ ]:
comparacao_prompts = pd.DataFrame([
    {"versao": "V1", "caracteres": len(prompt_v1), "componentes": int(verificar_componentes(prompt_v1)["presente"].sum())},
    {"versao": "V2", "caracteres": len(prompt_v2), "componentes": int(verificar_componentes(prompt_v2)["presente"].sum())},
    {"versao": "V3", "caracteres": len(prompt_v3), "componentes": int(verificar_componentes(prompt_v3)["presente"].sum())},
])
display(comparacao_prompts)


## 8. Teste de alucinação

O campo abaixo foi propositalmente omitido dos dados. Copie a V3 para a ferramenta autorizada e pergunte por ele. A resposta esperada é declarar que a informação não foi fornecida.


In [ ]:
pergunta_teste = f"Qual é {caso_grupo['informacao_ausente']}?"
print("Informação ausente:", caso_grupo["informacao_ausente"])
print("Pergunta para o teste:", pergunta_teste)
print("Resposta esperada: a informação não foi fornecida ou não há informação suficiente.")


### Registro do teste

**Resposta obtida:**  
Cole aqui.

**Inventou informação?** Sim / Não  
**O que precisa melhorar no prompt?**  
Escreva aqui.


## 9. Privacidade e validação humana


In [ ]:
checklist = carregar_csv("checklist_avaliacao_prompts.csv")
checklist["atendido"] = "PREENCHER"
display(checklist)


Antes de testar, confirme:

- os dados são sintéticos ou anonimizados;
- nenhum nome, documento ou dado sensível desnecessário será enviado;
- o modelo não recebeu autoridade para diagnosticar, prescrever ou decidir sozinho;
- existe uma pessoa responsável pela revisão;
- a saída será tratada como rascunho quando houver impacto sobre pessoas.


## 10. Atividade — IA ou não?

Classifique como REGRA, ML ou IA GENERATIVA:

1. detectar sobreposição de horários;
2. prever prioridade com histórico;
3. gerar resumo da agenda;
4. classificar comentário em tema;
5. verificar campo obrigatório vazio;
6. redigir descrição de dados estruturados.

**Respostas:** escreva aqui.


## 11. Exportando a documentação inicial


In [ ]:
documentacao = f"""# Aula 8 — Documentação de Prompt Engineering

## Funcionalidade
Caso: {caso_grupo['id_caso']} — {caso_grupo['projeto']}
Tarefa: {caso_grupo['tarefa']}

## Prompt V1
{prompt_v1}

### Resultado V1
[COLE A RESPOSTA]

### Problemas encontrados
[PREENCHA]

## Prompt V2
{prompt_v2}

### O que melhoramos?
[PREENCHA]

## Prompt V3
{prompt_v3}

### Por que esta versão é melhor?
[PREENCHA]

## Teste de informação ausente
Pergunta: {pergunta_teste}
Resposta obtida: [COLE A RESPOSTA]
Houve invenção? [SIM/NÃO]

## Segurança e privacidade
[PREENCHA]

## Validação humana
[PREENCHA QUEM REVISA E COMO]

## Conclusão
[PREENCHA]
"""

with open("04_ia_generativa_prompts.md", "w", encoding="utf-8") as arquivo:
    arquivo.write(documentacao)

pd.DataFrame({"versao": ["V1", "V2", "V3"], "prompt": [prompt_v1, prompt_v2, prompt_v3]}).to_csv(
    "prompts_v1_v2_v3.csv", index=False
)
print("Arquivos 04_ia_generativa_prompts.md e prompts_v1_v2_v3.csv gerados!")


## Checklist de entrega

- [x] descrição da funcionalidade;
- [x] prompts V1, V2 e V3;
- [ ] resultados reais dos três testes;
- [ ] problema encontrado e melhoria;
- [ ] teste de informação ausente;
- [ ] análise de alucinação;
- [ ] regras de segurança e privacidade;
- [ ] validação humana;
- [ ] conclusão.
